# GeoPlant — Single Predictor Run (imports framework)

In [3]:
# !pip install -r requirements.txt
import sys

sys.path.append("./")  # adjust to where you place this folder
from geoplant_xgb.config import ExperimentConfig, PredictorPairSpec
from geoplant_xgb.io_csv import (
    load_metadata_csv,
    load_predictor_pairs,
    build_features_from_meta_and_predictors_pair,
)
from geoplant_xgb.data import (
    build_wide_labels_from_long_metadata,
    align_features_with_labels,
    select_top_species,
    split_features_by_group,
)
from geoplant_xgb.model import (
    train_ovr,
    predict_scores,
    train_richness_estimator,
    estimate_topk,
)
import pandas as pd, numpy as np

/Users/lukaspicek/Documents/INRIA/GLC/GeoPlant/env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# Train/Test metadata (CSV). Train contains speciesId per surveyId; test usually doesn't.
TRAIN_METADATA_CSV = "../../metadata/PA_metadata_train.csv"
TEST_METADATA_CSV = "../../metadata/PA_metadata_test.csv"

# Single predictor family (bioclimatic) — separate TRAIN and TEST files
BIOCLIM_TRAIN_CSV = "./EnvironmentalValues/Climate/PA-train-bioclimatic.csv"
BIOCLIM_TEST_CSV = "./EnvironmentalValues/Climate/PA-test-bioclimatic.csv"

RESULTS_CSV_PATH = "xgb_single_bioclim_results.csv"
SUBMISSION_CSV_PATH = "submission_predictions.csv"

In [5]:
cfg = ExperimentConfig(
    predictor_pairs=[
        PredictorPairSpec(
            name="bioclim",
            group="climatic",
            prefix="clim_",
            train_path=BIOCLIM_TRAIN_CSV,
            test_path=BIOCLIM_TEST_CSV,
        )
    ],
    sample_id_col="survey_id",
    species_prefix="sp_",
    top_species_n=500,
    min_pos_per_species=5,
    xgb_params=dict(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        tree_method="hist",
        n_jobs=8,
        random_state=42,
        verbosity=0,
    ),
    early_stopping_rounds=30,
    ablations=[["climatic"]],
    use_richness_estimator=True,
    richness_offset=5,
)
train_meta = load_metadata_csv(TRAIN_METADATA_CSV, sample_id_col="surveyId")
test_meta = load_metadata_csv(TEST_METADATA_CSV, sample_id_col="surveyId")
train_pred, test_pred = load_predictor_pairs(cfg.predictor_pairs)
train_features, test_features = build_features_from_meta_and_predictors_pair(
    cfg, train_meta, test_meta, train_pred, test_pred
)

In [6]:
train_labels_wide = build_wide_labels_from_long_metadata(
    train_meta,
    survey_id_column="surveyId",
    species_id_column="speciesId",
    species_prefix=cfg.species_prefix,
)
train_feat_aligned, train_lab_aligned, species_cols_all = align_features_with_labels(
    train_features, train_labels_wide, survey_id_column=cfg.sample_id_col
)
# Build placeholder labels for test with same species columns
test_labels_placeholder = pd.DataFrame(
    {cfg.sample_id_col: test_features[cfg.sample_id_col]}
)
for c in species_cols_all:
    test_labels_placeholder[c] = 0
test_feat_aligned, test_lab_aligned, _ = align_features_with_labels(
    test_features, test_labels_placeholder, survey_id_column=cfg.sample_id_col
)

/var/folders/dk/gk7swfzx4vv6793v1g7wcl680000gn/T/ipykernel_32286/3263210242.py:15: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  test_labels_placeholder[c] = 0
/var/folders/dk/gk7swfzx4vv6793v1g7wcl680000gn/T/ipykernel_32286/3263210242.py:15: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  test_labels_placeholder[c] = 0
/var/folders/dk/gk7swfzx4vv6793v1g7wcl680000gn/T/ipykernel_32286/3263210242.py:15: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poo

In [7]:
# Select top species and train
top_species = select_top_species(train_lab_aligned, species_cols_all, cfg)
groups = split_features_by_group(train_feat_aligned, cfg)
clim_cols = groups["climatic"]
models = train_ovr(
    train_feat_aligned[clim_cols],
    train_lab_aligned.set_index(cfg.sample_id_col),
    top_species,
    cfg,
)
# Estimate Top-K
if cfg.use_richness_estimator:
    clf, edges, bin_to_avg = train_richness_estimator(
        train_feat_aligned[clim_cols],
        train_lab_aligned.set_index(cfg.sample_id_col),
        cfg,
        nbins=15,
    )
    topk_arr = estimate_topk(
        clf, edges, bin_to_avg, test_feat_aligned[clim_cols], offset=cfg.richness_offset
    )
else:
    topk_arr = cfg.fixed_top_k
# Export predictions
scores_test = predict_scores(models, test_feat_aligned[clim_cols], top_species)
order = np.argsort(-scores_test, axis=1)
pred_strings = []
for i in range(scores_test.shape[0]):
    k = int(topk_arr[i]) if hasattr(topk_arr, "__len__") else int(topk_arr)
    k = max(1, min(k, scores_test.shape[1]))
    idx = order[i, :k]
    ids = [top_species[j].replace("sp_", "") for j in idx]
    pred_strings.append(" ".join(ids))
submission = pd.DataFrame(
    {
        "surveyId": test_feat_aligned[cfg.sample_id_col].values,
        "predictions": pred_strings,
    }
)
submission.to_csv(SUBMISSION_CSV_PATH, index=False)
submission.head()

Training OVR models:   0%|          | 0/500 [00:00<?, ?it/s]

[0]	validation_0-logloss:0.60266
[1]	validation_0-logloss:0.59485
[2]	validation_0-logloss:0.58779
[3]	validation_0-logloss:0.58103
[4]	validation_0-logloss:0.57512
[5]	validation_0-logloss:0.56944
[6]	validation_0-logloss:0.56416
[7]	validation_0-logloss:0.55937
[8]	validation_0-logloss:0.55464
[9]	validation_0-logloss:0.55050
[10]	validation_0-logloss:0.54648
[11]	validation_0-logloss:0.54273
[12]	validation_0-logloss:0.53915
[13]	validation_0-logloss:0.53584
[14]	validation_0-logloss:0.53279
[15]	validation_0-logloss:0.53000
[16]	validation_0-logloss:0.52744
[17]	validation_0-logloss:0.52492
[18]	validation_0-logloss:0.52238
[19]	validation_0-logloss:0.52021
[20]	validation_0-logloss:0.51802
[21]	validation_0-logloss:0.51606
[22]	validation_0-logloss:0.51406
[23]	validation_0-logloss:0.51231
[24]	validation_0-logloss:0.51062
[25]	validation_0-logloss:0.50906
[26]	validation_0-logloss:0.50757
[27]	validation_0-logloss:0.50618
[28]	validation_0-logloss:0.50474
[29]	validation_0-loglos

Training OVR models:   0%|          | 1/500 [00:06<54:46,  6.59s/it]

[0]	validation_0-logloss:0.58375
[1]	validation_0-logloss:0.57499
[2]	validation_0-logloss:0.56688
[3]	validation_0-logloss:0.55943
[4]	validation_0-logloss:0.55264
[5]	validation_0-logloss:0.54636
[6]	validation_0-logloss:0.54060
[7]	validation_0-logloss:0.53499
[8]	validation_0-logloss:0.52994
[9]	validation_0-logloss:0.52534
[10]	validation_0-logloss:0.52115
[11]	validation_0-logloss:0.51682
[12]	validation_0-logloss:0.51304
[13]	validation_0-logloss:0.50921
[14]	validation_0-logloss:0.50593
[15]	validation_0-logloss:0.50268
[16]	validation_0-logloss:0.49984
[17]	validation_0-logloss:0.49704
[18]	validation_0-logloss:0.49429
[19]	validation_0-logloss:0.49184
[20]	validation_0-logloss:0.48945
[21]	validation_0-logloss:0.48720
[22]	validation_0-logloss:0.48511
[23]	validation_0-logloss:0.48321
[24]	validation_0-logloss:0.48128
[25]	validation_0-logloss:0.47943
[26]	validation_0-logloss:0.47778
[27]	validation_0-logloss:0.47617
[28]	validation_0-logloss:0.47478
[29]	validation_0-loglos

Training OVR models:   0%|          | 2/500 [00:12<53:42,  6.47s/it]

...

Training OVR models:   1%|          | 3/500 [00:19<54:03,  6.53s/it]

[0]	validation_0-logloss:0.52559
[1]	validation_0-logloss:0.51774
[2]	validation_0-logloss:0.51044
[3]	validation_0-logloss:0.50381
[4]	validation_0-logloss:0.49779
[5]	validation_0-logloss:0.49227
[6]	validation_0-logloss:0.48706
[7]	validation_0-logloss:0.48245
[8]	validation_0-logloss:0.47811
[9]	validation_0-logloss:0.47411
[10]	validation_0-logloss:0.47030
[11]	validation_0-logloss:0.46688
[12]	validation_0-logloss:0.46359
[13]	validation_0-logloss:0.46068
[14]	validation_0-logloss:0.45788
[15]	validation_0-logloss:0.45527
[16]	validation_0-logloss:0.45273
[17]	validation_0-logloss:0.45041
[18]	validation_0-logloss:0.44829
[19]	validation_0-logloss:0.44634
[20]	validation_0-logloss:0.44427
[21]	validation_0-logloss:0.44237
[22]	validation_0-logloss:0.44078
[23]	validation_0-logloss:0.43930
[24]	validation_0-logloss:0.43784
[25]	validation_0-logloss:0.43646
[26]	validation_0-logloss:0.43517
[27]	validation_0-logloss:0.43400
[28]	validation_0-logloss:0.43272
[29]	validation_0-loglos

Training OVR models:   1%|          | 4/500 [00:25<52:51,  6.39s/it]

[0]	validation_0-logloss:0.51591
[1]	validation_0-logloss:0.51078
[2]	validation_0-logloss:0.50609
[3]	validation_0-logloss:0.50231
[4]	validation_0-logloss:0.49868
[5]	validation_0-logloss:0.49547
[6]	validation_0-logloss:0.49200
[7]	validation_0-logloss:0.48929
[8]	validation_0-logloss:0.48639
[9]	validation_0-logloss:0.48360
[10]	validation_0-logloss:0.48125
[11]	validation_0-logloss:0.47901
[12]	validation_0-logloss:0.47699
[13]	validation_0-logloss:0.47500
[14]	validation_0-logloss:0.47258
[15]	validation_0-logloss:0.47052
[16]	validation_0-logloss:0.46877
[17]	validation_0-logloss:0.46725
[18]	validation_0-logloss:0.46584
[19]	validation_0-logloss:0.46443
[20]	validation_0-logloss:0.46302
[21]	validation_0-logloss:0.46181
[22]	validation_0-logloss:0.46060
[23]	validation_0-logloss:0.45904
[24]	validation_0-logloss:0.45762
[25]	validation_0-logloss:0.45665
[26]	validation_0-logloss:0.45550
[27]	validation_0-logloss:0.45459
[28]	validation_0-logloss:0.45367
[29]	validation_0-loglos

KeyboardInterrupt: 

In [ ]:
# Train-set sanity metrics (not a valid test estimate)
scores_train = predict_scores(models, train_feat_aligned[clim_cols], top_species)
from geoplant_xgb.metrics import sample_f1_at_k, sample_recall_at_k, macro_auc

fs1 = sample_f1_at_k(
    train_lab_aligned[top_species].values.astype(int), scores_train, k=25
)
rec = sample_recall_at_k(
    train_lab_aligned[top_species].values.astype(int), scores_train, k=25
)
auc = macro_auc(train_lab_aligned[top_species].values.astype(int), scores_train)
import pandas as pd

results = pd.DataFrame(
    {
        "groups": ["climatic"],
        "n_features": [len(clim_cols)],
        "n_species": [len(top_species)],
        "AUC": [auc],
        "Recall": [rec],
        "Fs1": [fs1],
        "TopK": ["per-sample"],
    }
)
results.to_csv(RESULTS_CSV_PATH, index=False)
results